# Практика · RNN для тексту> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) · Домашнє: [homework.html](homework.html)> ⏱ **Зошит навчає три мовні моделі на повному корпусі.** Заміряно> `check_notebook.py` наодинці: близько **пʼяти з половиною хвилин** процесорного часу> на чотирьох ядрах без відеокарти (реальний час залежить від того, чим ще> зайнята машина). Це не зависання — це навчання, і воно і є предметом теми.Що зробимо:1. зберемо корпус і поділимо його так само, як у темі 16;2. заміряємо, скільки контексту n-грамна модель узагалі бачила;3. підберемо добавку згладжування **на відкладеній вибірці**, а не на перевірній;4. **напишемо крок рекурентності руками** й переконаємось, що він збігається з `nn.RNN`;5. порахуємо, скільки ваг економлять спільні ваги;6. навчимо RNN-мовну модель на трьох зернах і порівняємо з n-грамами;7. заміряємо, як далеко сягає памʼять стану.

## 0 · СередовищеГрабля курсу: **кількість потоків фіксуємо до імпорту numpy**. Без цього процесорнийчас бреше в рази — потоки OpenMP крутяться в очікуванні, і це очікування рахуєтьсяяк робота.

In [ ]:
import os
# ці чотири рядки мусять стояти ДО імпорту numpy і torch, інакше
# time.process_time() покаже час усіх потоків, що чекають один на одного
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

import sys, re, glob, gettext, math, random, time
from collections import Counter, defaultdict
import numpy as np
import torch
import torch.nn as nn

torch.set_num_threads(1)          # те саме для torch, уже після імпорту

print('python     ', sys.version.split()[0])
print('numpy      ', np.__version__)
print('torch      ', torch.__version__)
print('потоків    ', torch.get_num_threads())

## 1 · КорпусТой самий, що й у всьому курсі: українські переклади інтерфейсів програм, яківже лежать у системі. Токенізатор — канонічний для блоку.

In [ ]:
TOKEN_PATTERN = r"[а-яїієґ]+(?:['ʼ’][а-яїієґ]+)*"    # канон блоку, після lowercase
_token_re = re.compile(TOKEN_PATTERN)

def tokenize(text):
    """Ріже рядок на слова. Апостроф — звʼязка всередині слова, а не межа."""
    return _token_re.findall(text.lower())

def load_corpus():
    """Читає всі українські каталоги перекладів, що є на цій машині."""
    docs = []
    for path in sorted(glob.glob('/usr/share/locale/uk/LC_MESSAGES/*.mo')):
        try:
            with open(path, 'rb') as f:
                catalog = gettext.GNUTranslations(f)
            for src, dst in catalog._catalog.items():
                if isinstance(src, str) and isinstance(dst, str) \
                   and len(dst) > 30 and 'Project-Id' not in dst:
                    docs.append(dst)
        except Exception:
            pass
    return docs

documents = load_corpus()
sentences = [tokenize(d) for d in documents]
lengths = np.array([len(s) for s in sentences])

print('документів   ', len(documents))
print('слововживань ', int(lengths.sum()))
print('словоформ    ', len({w for s in sentences for w in s}))
print('медіана      ', int(np.median(lengths)), 'слів')
print('90-й перцентиль', int(np.percentile(lengths, 90)))
print('найдовше     ', int(lengths.max()))

Медіана **6 слів** — головна властивість цього корпусу для теми. Довгої історії,заради якої й вигадали моделі зі станом, тут майже немає, і це відібʼється на всіхподальших числах.## 2 · Поділ і словникЛишаємо речення від 2 до 30 слів, ділимо 90/10 із зерном 0. Словник будуємо**лише за навчальною частиною** — інакше модель побачила б слова перевірної.

In [ ]:
kept = [s for s in sentences if 2 <= len(s) <= 30]

shuffled = list(kept)
random.Random(0).shuffle(shuffled)          # зерно 0: поділ однаковий у всіх
n_train = int(len(shuffled) * 0.9)
train_words, val_words = shuffled[:n_train], shuffled[n_train:]

MIN_COUNT = 5
counts = Counter(w for s in train_words for w in s)
vocab = ['<pad>', '<eos>', '<unk>'] + sorted(w for w, c in counts.items() if c >= MIN_COUNT)
word_to_id = {w: i for i, w in enumerate(vocab)}
PAD, EOS, UNK = 0, 1, 2
V = len(vocab)

def encode(sents):
    return [[word_to_id.get(w, UNK) for w in s] for s in sents]

train = encode(train_words)
val = encode(val_words)

# кожне речення дає стільки передбачень, скільки в ньому слів, плюс одне на <eos>
targets_train = sum(len(s) + 1 for s in train)
targets_val = sum(len(s) + 1 for s in val)

print('після відсіву   ', len(kept), 'речень')
print('навчальних      ', len(train), '- цілей', targets_train)
print('перевірних      ', len(val), '- цілей', targets_val)
print('словник         ', V, '(разом із <pad>, <eos>, <unk>)')
print('частка <unk> у перевірній  %.4f'
      % (sum(1 for s in val for w in s if w == UNK) / sum(len(s) for s in val)))

## 3 · Скільки контексту n-грамна модель узагалі бачилаГоловне питання до лічильникових моделей: чи траплявся їм у навчанні той контекст,який вони зустрінуть на перевірці. Якщо ні — лічильник порожній, і модель можевідповісти лише згладжуванням.Рахуємо дві різні речі:* **контекст** — те, за чим модель шукає в таблиці (для біграми це одне попереднє слово);* **повна n-грама** — контекст разом із наступним словом, тобто те, що модель мусить  порахувати насправді.Ці числа малює **інтерактив 1** лекції.

In [ ]:
t0 = time.process_time()
ORDERS = [1, 2, 3, 4, 5]
context_seen = {n: set() for n in ORDERS}
ngram_seen = {n: set() for n in ORDERS}

for s in train:
    padded = [EOS] * 4 + s + [EOS]           # чотири <eos> зліва: вистачає на n до 5
    for i in range(4, len(padded)):
        for n in ORDERS:
            context_seen[n].add(tuple(padded[i - n + 1:i]))
            ngram_seen[n].add(tuple(padded[i - n + 1:i + 1]))

coverage = []
for n in ORDERS:
    hit_ctx = hit_ng = total = 0
    for s in val:
        padded = [EOS] * 4 + s + [EOS]
        for i in range(4, len(padded)):
            total += 1
            if tuple(padded[i - n + 1:i]) in context_seen[n]:
                hit_ctx += 1
            if tuple(padded[i - n + 1:i + 1]) in ngram_seen[n]:
                hit_ng += 1
    coverage.append((n, len(context_seen[n]), len(ngram_seen[n]),
                     hit_ctx / total, hit_ng / total))

print('n | різних контекстів | різних n-грам | бачили контекст | бачили n-граму | прикладів на контекст')
for n, nc, nn_, hc, hn in coverage:
    print('%d | %17d | %13d | %14.4f | %14.4f | %8.1f'
          % (n, nc, nn_, hc, hn, targets_train / nc))
print('процесорних секунд: %.1f' % (time.process_time() - t0))

Читається так: біграма бачила свій контекст **завжди** (одне слово, різних усьогокілька тисяч), а от повну біграму «слово → наступне слово» — уже ні. Для пʼятиграмибільшість перевірних позицій модель бачить уперше.І окремо — розмір самих моделей: біграма зберігає стільки чисел, скільки в неїрізних контекстів плюс різних пар.

In [ ]:
bigram_size = coverage[1][1] + coverage[1][2]     # контексти + пари
trigram_size = coverage[2][1] + coverage[2][2]
print('чисел у біграмній моделі  ', bigram_size)
print('чисел у триграмній моделі ', trigram_size)

## 4 · База: n-грама з добавкою, підібраною чесноЗгладжування «додати α»: до кожного лічильника додаємо α, а до знаменника — α·V.Питання в тому, звідки взяти α. Підбирати його **на перевірній частині** означаєпідглядати у відповідь, тому відрізаємо з навчальної частини **відкладену** вибірку,обираємо α на ній і лише потім міряємо на перевірній.

In [ ]:
holdout_index = list(range(len(train)))
random.Random(11).shuffle(holdout_index)
n_holdout = int(len(train) * 0.05)
holdout = [train[i] for i in holdout_index[:n_holdout]]
train_minus = [train[i] for i in holdout_index[n_holdout:]]
print('відкладена вибірка:', n_holdout, 'речень')

def fit_ngram(data, order):
    """Рахує, скільки разів траплявся кожен контекст і кожна повна n-грама."""
    ctx = Counter(); nxt = Counter()
    for s in data:
        padded = [EOS] * (order - 1) + s + [EOS]
        for i in range(order - 1, len(padded)):
            c = tuple(padded[i - order + 1:i])
            ctx[c] += 1
            nxt[(c, padded[i])] += 1
    return ctx, nxt

def ngram_perplexity(ctx, nxt, data, order, alpha):
    """Перплексія = e у степені середньої мінус-логарифмічної ймовірності."""
    total, n = 0.0, 0
    for s in data:
        padded = [EOS] * (order - 1) + s + [EOS]
        for i in range(order - 1, len(padded)):
            c = tuple(padded[i - order + 1:i])
            p = (nxt.get((c, padded[i]), 0) + alpha) / (ctx.get(c, 0) + alpha * V)
            total -= math.log(p); n += 1
    return math.exp(total / n)

ALPHAS = [1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005,
          0.002, 0.001, 0.0005, 0.0002, 0.0001]

t0 = time.process_time()
ngram_result = {}
for order in (2, 3):
    ctx_h, nxt_h = fit_ngram(train_minus, order)      # для добору добавки
    ctx_f, nxt_f = fit_ngram(train, order)            # для остаточного заміру
    on_holdout = [ngram_perplexity(ctx_h, nxt_h, holdout, order, a) for a in ALPHAS]
    on_val = [ngram_perplexity(ctx_f, nxt_f, val, order, a) for a in ALPHAS]
    best = ALPHAS[int(np.argmin(on_holdout))]
    ngram_result[order] = dict(holdout=on_holdout, val=on_val, best=best,
                               best_val=on_val[ALPHAS.index(best)], ctx=ctx_f, nxt=nxt_f)
    print('\nпорядок %d' % order)
    print('  добавка |   відкладена |    перевірна')
    for a, h, v in zip(ALPHAS, on_holdout, on_val):
        print('  %-7g | %12.4f | %12.4f' % (a, h, v))
    print('  найкраща добавка за відкладеною: %g  ->  на перевірній %.4f'
          % (best, on_val[ALPHAS.index(best)]))
print('\nпроцесорних секунд: %.1f' % (time.process_time() - t0))

Ці дві криві малює **інтерактив 5** лекції. Дві речі, які варто побачити одразу:* добавка 0.1, узята в темі 16 як кругле число, — **не найкраща**;* триграма гірша за біграму, хоч і бачить удвічі більше контексту.Для повноти — дві крайні точки шкали: модель, яка нічого не знає, і модель, яказнає лише частоти окремих слів.

In [ ]:
ctx_uni, nxt_uni = fit_ngram(train, 1)
unigram_ppl = ngram_perplexity(ctx_uni, nxt_uni, val, 1, 1.0)
print('рівномірна (навмання зі словника) ', V)
print('уніграма з добавкою 1             %.4f' % unigram_ppl)
print('біграма з добавкою 0.1            %.4f' % ngram_result[2]['val'][ALPHAS.index(0.1)])
print('біграма з добавкою 1              %.4f' % ngram_result[2]['val'][ALPHAS.index(1.0)])
print('найкраща біграма (добавка %g)  %.4f' % (ngram_result[2]['best'], ngram_result[2]['best_val']))

t0 = time.process_time()
_ = fit_ngram(train, 2)
bigram_seconds = time.process_time() - t0
print('\nпобудова біграми коштує %.2f с процесорних' % bigram_seconds)

## 5 · Крок рекурентності, написаний рукамиГоловна перевірка теми. Формула одна:    h_t = tanh(W_x · x_t + W_h · h_(t-1) + b)Беремо `nn.RNN`, забираємо **його власні** ваги й проганяємо ті самі дані звичайнимциклом. Якщо всередині бібліотечного шару стоїть та сама формула, результати маютьзбігтися з точністю арифметики `float32`.

In [ ]:
def manual_rnn(cell, x):
    """Наш власний прохід рекурентної клітинки. Нічого, крім формули з лекції."""
    W_x = cell.weight_ih_l0        # (H, E) - множники для слова
    W_h = cell.weight_hh_l0        # (H, H) - множники для памʼяті
    b_x = cell.bias_ih_l0          # у nn.RNN два зсуви; разом вони рівносильні одному
    b_h = cell.bias_hh_l0
    batch, steps, _ = x.shape
    h = torch.zeros(batch, cell.hidden_size)
    outputs = []
    for t in range(steps):
        # ось увесь рекурентний крок: слово, памʼять, зсув, притискання
        h = torch.tanh(x[:, t, :] @ W_x.T + h @ W_h.T + b_x + b_h)
        outputs.append(h)
    return torch.stack(outputs, dim=1)

def compare_manual(E, H, steps, batch, seed):
    torch.manual_seed(seed)
    cell = nn.RNN(E, H, batch_first=True)
    x = torch.randn(batch, steps, E)
    ours = manual_rnn(cell, x)
    theirs, _ = cell(x)
    return (ours - theirs).abs().max().item(), bool(torch.equal(ours, theirs))

print('машинний епсилон float32: %.6e' % torch.finfo(torch.float32).eps)
print()
print('іграшковий розмір (вхід 5, стан 4, 7 кроків):')
toy_diffs = []
for seed in (0, 1, 2):
    d, same = compare_manual(5, 4, 7, 3, seed)
    toy_diffs.append(d)
    print('   зерно %d: найбільша розбіжність %.6e, побітово однакові: %s' % (seed, d, same))
print()
print('реальний розмір моделі (вхід 64, стан 128, 30 кроків):')
real_diffs = []
for seed in (0, 1, 2):
    d, same = compare_manual(64, 128, 30, 64, seed)
    real_diffs.append(d)
    print('   зерно %d: найбільша розбіжність %.6e, побітово однакові: %s' % (seed, d, same))

Перевірка, заради якої все й робилось: **наша реалізація дорівнює бібліотечній**.Побітового збігу тут не буває — бібліотека складає доданки в іншому порядку, — томуправильний критерій звучить «різниця не більша за одиницю останнього розряду».

In [ ]:
torch.manual_seed(0)
cell = nn.RNN(5, 4, batch_first=True)
x = torch.randn(3, 7, 5)
ours = manual_rnn(cell, x)
theirs, _ = cell(x)

assert torch.allclose(ours, theirs, atol=1e-6, rtol=0), 'розрахунок розійшовся!'
print('✅ ручний крок збігається з nn.RNN')
print('   найбільша розбіжність %.6e = %.2f машинного епсилона float32'
      % (toy_diffs[0], toy_diffs[0] / torch.finfo(torch.float32).eps))

## 6 · Спільні ваги: скільки це чиселРекурентна клітинка застосовує **один** набір ваг на всіх кроках. Порахуймо, чим цевідрізняється від мережі, у якої на кожен крок свій набір.

In [ ]:
EMB, HID = 64, 128
T_MAX = 30                                   # найдовше речення в нашому наборі

weights_embedding = V * EMB
weights_cell = HID * EMB + HID * HID + 2 * HID    # W_x, W_h і два зсуви
weights_output = HID * V + V

shared_total = weights_embedding + weights_cell + weights_output
separate_total = weights_embedding + weights_cell * T_MAX + weights_output

print('ембединги слів      ', weights_embedding)
print('рекурентна клітинка ', weights_cell,
      '= %d (W_x) + %d (W_h) + %d (зсуви)' % (HID * EMB, HID * HID, 2 * HID))
print('вихідний шар        ', weights_output)
print()
print('RNN зі спільними вагами          ', shared_total)
print('окремі ваги на кожен із %d кроків ' % T_MAX, separate_total)
print('клітинки, якби ваги були окремі  ', weights_cell * T_MAX)
print('економія в клітинці              ', weights_cell * T_MAX - weights_cell,
      '(рівно у %d разів)' % T_MAX)
print('уся мережа більша в              %.4f раза' % (separate_total / shared_total))
print('частка клітинки у вагах RNN      %.2f %%' % (100 * weights_cell / shared_total))

Економія в клітинці — тридцятикратна, а вся мережа росте лише в півтора раза: у мовноїмоделі більшість ваг витрачено на словникові шари. Тому головний довід на користьспільних ваг **не памʼять**, а два інші: модель не прибита до довжини, на якій вчилась,і кожне слово корпусу є прикладом для того самого правила.## 7 · Іграшкова клітинка: один крок на числахЩоб побачити формулу очима, візьмемо стан із чотирьох чисел і ембединги з трьох.Множники — випадкові, округлені до сотих, щоб їх можна було перевірити на папері.Ці числа малює **інтерактив 2** лекції.

In [ ]:
gen = np.random.default_rng(7)
toy_emb = np.round(gen.uniform(-1, 1, (4, 3)), 2)     # 4 іграшкові слова по 3 числа
toy_Wx = np.round(gen.uniform(-1, 1, (4, 3)), 2)      # стан 4, вхід 3
toy_Wh = np.round(gen.uniform(-0.8, 0.8, (4, 4)), 2)
toy_b = np.round(gen.uniform(-0.3, 0.3, 4), 2)

h = np.zeros(4)
for t in range(4):
    from_word = toy_Wx @ toy_emb[t]          # внесок поточного слова
    from_memory = toy_Wh @ h                 # внесок памʼяті: єдине місце, де живе історія
    z = from_word + from_memory + toy_b
    h = np.tanh(z)
    print('крок %d' % (t + 1))
    print('   від слова  ', np.round(from_word, 4))
    print('   від памʼяті ', np.round(from_memory, 4))
    print('   сума        ', np.round(z, 4))
    print('   h = tanh    ', np.round(h, 4))
    print('   сила слова %.4f, сила памʼяті %.4f'
          % (np.linalg.norm(from_word), np.linalg.norm(from_memory)))

## 8 · Мовна модель на RNNТри шари: таблиця ембедингів, рекурентна клітинка, лінійний шар на весь словник.Службовий `<eos>` подаємо на вхід перед першим словом і чекаємо на виході післяостаннього — так одна модель уміє і почати речення, і закінчити його.

In [ ]:
class RNNLanguageModel(nn.Module):
    def __init__(self, vocab_size, emb_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_size, padding_idx=PAD)
        self.cell = nn.RNN(emb_size, hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        states, _ = self.cell(self.embedding(x))
        return self.output(states), states

def make_batch(chunk):
    """Пачка з речень різної довжини: короткі доповнюємо нулями до найдовшого."""
    width = max(len(s) for s in chunk) + 1
    x = torch.zeros(len(chunk), width, dtype=torch.long)
    y = torch.zeros(len(chunk), width, dtype=torch.long)
    for row, s in enumerate(chunk):
        sequence = [EOS] + s + [EOS]
        x[row, :len(s) + 1] = torch.tensor(sequence[:-1])
        y[row, :len(s) + 1] = torch.tensor(sequence[1:])
    return x, y

def random_groups(data, size, seed):
    """Пачки з випадкових речень - найпростіший спосіб, і найдорожчий."""
    index = list(range(len(data)))
    random.Random(seed).shuffle(index)
    return [[data[j] for j in index[i:i + size]] for i in range(0, len(index), size)]

def length_groups(data, size, seed):
    """Пачки з речень схожої довжини: сортуємо за довжиною, а самі пачки перемішуємо."""
    index = sorted(range(len(data)), key=lambda j: (len(data[j]), j))
    groups = [[data[j] for j in index[i:i + size]] for i in range(0, len(index), size)]
    if seed is not None:
        random.Random(seed).shuffle(groups)
    return groups

print('модель:', RNNLanguageModel(V, EMB, HID))

Перш ніж навчати — одна деталь, яка коштує втричі. Медіана нашого речення 6 слів,а найдовше має 30. Якщо збирати пачку з випадкових речень, у неї майже завждипотрапить одне довге, і всі інші доведеться доповнювати нулями до нього.

In [ ]:
def useful_share(groups):
    """Яка частка клітинок пачки несе справжнє слово, а не заповнювач."""
    cells = sum(len(g) * (max(len(s) for s in g) + 1) for g in groups)
    return cells, targets_train / cells

seconds_for = {}
for name, groups in (('випадкові речення', random_groups(train, 64, 0)),
                     ('речення схожої довжини', length_groups(train, 64, 0))):
    cells, share = useful_share(groups)
    torch.manual_seed(0)
    probe = RNNLanguageModel(V, EMB, HID)
    optimizer = torch.optim.Adam(probe.parameters(), lr=2e-3)
    t0 = time.process_time()
    for group in groups[:15]:
        x, y = make_batch(group)
        optimizer.zero_grad()
        logits, _ = probe(x)
        nn.functional.cross_entropy(logits.reshape(-1, V), y.reshape(-1),
                                    ignore_index=PAD).backward()
        optimizer.step()
    seconds_for[name] = time.process_time() - t0
    print('%-24s клітинок %8d, справжніх %6.2f %%, 15 пачок %.2f с'
          % (name, cells, 100 * share, seconds_for[name]))
print('пачки з речень схожої довжини швидші в %.2f раза'
      % (seconds_for['випадкові речення'] / seconds_for['речення схожої довжини']))

Дві третини роботи йшли в порожнечу. Далі вчимось на пачках із речень схожої довжини —модель бачить те саме, а часу йде втричі менше.## 9 · Навчання: три зернаКонфігурація блоку: ембединг 64, стан 128, пачка 64, `Adam` зі швидкістю 0.002,**одна епоха**. Три зерна — бо різниця, менша за розкид, не є різницею.Це найдовша клітинка зошита: приблизно **півтори хвилини процесорного часу на зерно**.

In [ ]:
def evaluate(model, data):
    """Перплексія на перевірній частині плюс розбивка за позицією в реченні."""
    model.eval()
    total, n = 0.0, 0
    loss_at = np.zeros(31); count_at = np.zeros(31)
    with torch.no_grad():
        for group in length_groups(data, 128, None):
            x, y = make_batch(group)
            logits, _ = model(x)
            each = nn.functional.cross_entropy(
                logits.reshape(-1, V), y.reshape(-1),
                ignore_index=PAD, reduction='none').reshape(y.shape)
            mask = (y != PAD).float()
            total += float((each * mask).sum()); n += int(mask.sum())
            for p in range(min(31, y.shape[1])):
                loss_at[p] += float((each[:, p] * mask[:, p]).sum())
                count_at[p] += float(mask[:, p].sum())
    return math.exp(total / n), loss_at, count_at

def train_one(seed):
    torch.manual_seed(seed)
    model = RNNLanguageModel(V, EMB, HID)
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)
    model.train()
    t0 = time.process_time()
    for group in length_groups(train, 64, seed):
        x, y = make_batch(group)
        optimizer.zero_grad()
        logits, _ = model(x)
        nn.functional.cross_entropy(logits.reshape(-1, V), y.reshape(-1),
                                    ignore_index=PAD).backward()
        optimizer.step()
    seconds = time.process_time() - t0
    return model, seconds

models, rnn_ppl, rnn_time = [], [], []
loss_total = np.zeros(31); count_total = np.zeros(31)
for seed in (0, 1, 2):
    model, seconds = train_one(seed)
    ppl, la, ca = evaluate(model, val)
    models.append(model); rnn_ppl.append(ppl); rnn_time.append(seconds)
    loss_total += la; count_total += ca
    print('зерно %d: перплексія %.4f, навчання %.2f с процесорних' % (seed, ppl, seconds))

print()
print('RNN: %.4f ±%.4f  (від %.4f до %.4f)'
      % (np.mean(rnn_ppl), np.std(rnn_ppl, ddof=1), min(rnn_ppl), max(rnn_ppl)))
print('час: %.2f ±%.2f с на зерно' % (np.mean(rnn_time), np.std(rnn_time, ddof=1)))

## 10 · Підсумкове порівнянняТепер зведімо все в одну таблицю — і подивімось, що з цього виходить.

In [ ]:
best_alpha = ngram_result[2]['best']
rows = [
    ('рівномірна (нічого не знає)', float(V), None),
    ('уніграма, добавка 1', unigram_ppl, None),
    ('біграма, добавка 1', ngram_result[2]['val'][ALPHAS.index(1.0)], bigram_seconds),
    ('біграма, добавка 0.1 (база теми 16)', ngram_result[2]['val'][ALPHAS.index(0.1)], bigram_seconds),
    ('біграма, добавка %g (найкраща)' % best_alpha, ngram_result[2]['best_val'], bigram_seconds),
    ('триграма, добавка %g (найкраща)' % ngram_result[3]['best'], ngram_result[3]['best_val'], None),
    ('RNN, стан 128, одна епоха', float(np.mean(rnn_ppl)), float(np.mean(rnn_time))),
]
print('%-38s %12s' % ('модель', 'перплексія'))
for name, value, _ in rows:
    print('%-38s %12.4f' % (name, value))

print()
print('найгірша біграма мінус найкраща:      %.2f пункту'
      % (ngram_result[2]['val'][ALPHAS.index(1.0)] - ngram_result[2]['best_val']))
print('база теми 16 мінус RNN:               %.2f пункту'
      % (ngram_result[2]['val'][ALPHAS.index(0.1)] - np.mean(rnn_ppl)))
print('RNN мінус найкраща біграма:           %.2f пункту'
      % (np.mean(rnn_ppl) - ngram_result[2]['best_val']))
print('налаштування бази важить у %.1f раза більше за зміну методу'
      % ((ngram_result[2]['val'][ALPHAS.index(1.0)] - ngram_result[2]['best_val'])
         / (ngram_result[2]['val'][ALPHAS.index(0.1)] - np.mean(rnn_ppl))))
print('RNN дорожча за біграму у %.0f разів' % (np.mean(rnn_time) / bigram_seconds))

Це і є головне число теми, і воно незручне: **добре налаштована біграма перемагаєнашу RNN**. Порівняння нечесне — біграма дістала ручку, яку ми покрутили тринадцятьразів, а мережа одну конфігурацію без добору, — і саме в цьому урок.## 11 · Перплексія за позицією в реченніПерше слово доводиться називати без жодного контексту, десяте — маючи девʼять.Якщо стан справді накопичує історію, різниця з біграмою має залежати від позиції.

In [ ]:
def bigram_by_position(alpha):
    ctx = ngram_result[2]['ctx']; nxt = ngram_result[2]['nxt']
    loss_at = np.zeros(31); count_at = np.zeros(31)
    for s in val:
        sequence = [EOS] + s + [EOS]
        for i in range(1, len(sequence)):
            if i - 1 >= 31:
                break
            p = (nxt.get(((sequence[i - 1],), sequence[i]), 0) + alpha) \
                / (ctx.get((sequence[i - 1],), 0) + alpha * V)
            loss_at[i - 1] -= math.log(p); count_at[i - 1] += 1
    return loss_at, count_at

rnn_at = [math.exp(loss_total[p] / count_total[p]) if count_total[p] > 100 else None
          for p in range(31)]
bi_best_l, bi_best_c = bigram_by_position(best_alpha)
bi_naive_l, bi_naive_c = bigram_by_position(0.1)
bi_best_at = [math.exp(bi_best_l[p] / bi_best_c[p]) if bi_best_c[p] > 100 else None for p in range(31)]
bi_naive_at = [math.exp(bi_naive_l[p] / bi_naive_c[p]) if bi_naive_c[p] > 100 else None for p in range(31)]

print('позиція |    RNN | біграма %-5g | біграма 0.1 | RNN / найкраща біграма | випадків'
      % best_alpha)
for p in range(14):
    if rnn_at[p] is None:
        continue
    print('%7d | %6.1f | %13.1f | %11.1f | %22.2f | %8d'
          % (p + 1, rnn_at[p], bi_best_at[p], bi_naive_at[p],
             rnn_at[p] / bi_best_at[p], int(count_total[p])))

## 12 · Дві моделі йдуть реченням поручОдне число на весь корпус не каже, **де саме** моделі поводяться інакше. Візьмімоконкретні перевірні речення й подивімось на кожну позицію: що пропонує RNN, щопропонує біграма і яку ймовірність кожна дала справжньому наступному слову.Ці дані малює **інтерактив 6** лекції.

In [ ]:
by_previous = defaultdict(list)
for (context, word), k in ngram_result[2]['nxt'].items():
    by_previous[context[0]].append((word, k))

def bigram_row(previous, true_word):
    """Трійка найімовірніших наступних слів за біграмою і ймовірність справжнього."""
    denominator = ngram_result[2]['ctx'].get((previous,), 0) + best_alpha * V
    top = sorted(by_previous.get(previous, []), key=lambda t: -t[1])[:3]
    top_words = [(vocab[w], (k + best_alpha) / denominator) for w, k in top]
    p_true = (ngram_result[2]['nxt'].get(((previous,), true_word), 0) + best_alpha) / denominator
    return top_words, p_true

candidates = [s for s in val if 6 <= len(s) <= 9 and UNK not in s]
chosen = [candidates[i] for i in (9, 22, 4, 1)]

for s in chosen:
    print('«%s»' % ' '.join(vocab[w] for w in s))
    x = torch.tensor([[EOS] + s], dtype=torch.long)
    with torch.no_grad():
        logits, states = models[0](x)
        probs = torch.softmax(logits[0], dim=-1)
    inputs = [EOS] + s
    truths = s + [EOS]
    for t in range(len(inputs)):
        values, ids = probs[t].topk(3)
        rnn_top = ', '.join('%s %.3f' % (vocab[int(j)], float(p)) for p, j in zip(values, ids))
        bi_top, p_bi = bigram_row(inputs[t], truths[t])
        bi_text = ', '.join('%s %.3f' % (w, p) for w, p in bi_top)
        p_rnn = float(probs[t, truths[t]])
        print('   після «%s» -> справжнє «%s»' % (vocab[inputs[t]], vocab[truths[t]]))
        print('      RNN     %.4f  | %s' % (p_rnn, rnn_top))
        print('      біграма %.4f  | %s' % (p_bi, bi_text))
    print()

## 13 · Підсумок: числа, які називає лекціяОстанні кілька клітинок друкували все з чотирма знаками — так зручно звіряти. Лекціяназиває ті самі числа коротше, тож зведімо їх в один список у тій самій точності,у якій вони стоять у тексті.

In [ ]:
print('перплексія на перевірній частині')
print('   рівномірна                       %d' % V)
print('   уніграма, добавка 1              %.2f' % unigram_ppl)
print('   біграма, добавка 1               %.2f' % ngram_result[2]['val'][ALPHAS.index(1.0)])
print('   біграма, добавка 0.1             %.2f' % ngram_result[2]['val'][ALPHAS.index(0.1)])
print('   біграма, добавка %-5g найкраща   %.2f' % (best_alpha, ngram_result[2]['best_val']))
print('   триграма, добавка %-5g найкраща  %.2f' % (ngram_result[3]['best'], ngram_result[3]['best_val']))
print('   RNN, три зерна                   %.2f ±%.2f  (%.2f, %.2f, %.2f)'
      % (np.mean(rnn_ppl), np.std(rnn_ppl, ddof=1), *sorted(round(p, 2) for p in rnn_ppl)))
print()
print('ручний крок проти nn.RNN')
print('   іграшковий розмір   %.2f · 10 у мінус восьмій' % (toy_diffs[0] * 1e8))
print('   реальний розмір     від %.2f до %.2f · 10 у мінус сьомій'
      % (min(real_diffs) * 1e7, max(real_diffs) * 1e7))
print()
print('частка перевірних позицій, яких модель НЕ бачила (сто відсотків мінус покриття)')
for n, nc, nn_, hc, hn in coverage:
    print('   n=%d: контексту не було %.2f %%, повної n-грами не було %.2f %%'
          % (n, 100 * (1 - hc), 100 * (1 - hn)))
print()
print('ваги: клітинка спільна %d, клітинка окремо на 30 кроків %d, уся RNN %d'
      % (weights_cell, weights_cell * T_MAX, shared_total))

## 14 · Як далеко сягає памʼятьПрямий дослід: подаємо моделі перші десять слів, дивимось, що вона передбачаєодинадцятим, потім **підмінюємо одне слово** на відстані `k` і дивимось, чи зміниласьвідповідь. Два виміри: наскільки зсунувся розподіл на 8 440 слів і наскількизсунувся сам вектор стану. Ці числа малює **інтерактив 7** лекції.

In [ ]:
long_sentences = [s for s in val if len(s) >= 10][:200]
print('довгих перевірних речень узято:', len(long_sentences))

x_base = torch.zeros(len(long_sentences), 11, dtype=torch.long)
for row, s in enumerate(long_sentences):
    x_base[row] = torch.tensor([EOS] + s[:10])

DISTANCES = [1, 2, 3, 4, 5, 6, 7, 8]
picker = random.Random(3)
# підмінне слово для кожної відстані фіксуємо заздалегідь: усі три мережі
# мають дістати ту саму підміну, інакше розкид міряв би не мережі, а випадковість
replacement = {k: [picker.randrange(3, V) for _ in long_sentences] for k in DISTANCES}

shift_prediction = {k: [] for k in DISTANCES}
shift_state = {k: [] for k in DISTANCES}
for model in models:
    with torch.no_grad():
        logits, states = model(x_base)
        p_base = torch.softmax(logits[:, -1, :], dim=-1)
        h_base = states[:, -1, :]
    for k in DISTANCES:
        x_changed = x_base.clone()
        x_changed[:, 11 - k] = torch.tensor(replacement[k])
        with torch.no_grad():
            logits2, states2 = model(x_changed)
            p_new = torch.softmax(logits2[:, -1, :], dim=-1)
            h_new = states2[:, -1, :]
        shift_prediction[k].append(float((0.5 * (p_new - p_base).abs().sum(-1)).mean()))
        shift_state[k].append(float(((h_new - h_base).norm(dim=-1) / h_base.norm(dim=-1)).mean()))

print()
print('відстань | зсув відповіді | розкид | зсув стану | частка від k=1')
base_shift = np.mean(shift_prediction[1])
for k in DISTANCES:
    m = np.mean(shift_prediction[k])
    print('%8d | %14.4f | ±%.4f | %10.4f | %13.1f %%'
          % (k, m, np.std(shift_prediction[k], ddof=1), np.mean(shift_state[k]),
             100 * m / base_shift))

Вплив спадає приблизно **вдвічі з кожним кроком назад**. Це і є профіль памʼятізвичайної рекурентної клітинки: вона памʼятає далі за біграму (у якої на відстані 2вплив дорівнює рівно нулю), але не набагато далі. Чому саме так і що з цим роблятьгейти — тема [18 · LSTM і GRU](../18-lstm-gru/lecture.html).---## Завдання### 🟢 Рівень 1 — БазаПовтори перевірку «наш крок = `nn.RNN`», але для **двошарової** мережі(`nn.RNN(..., num_layers=2)`). Другий шар отримує на вхід стан першого.**Зроблено, якщо:** `torch.allclose` проходить із `atol=1e-6`, і ти можеш назвати,які саме ваги (`weight_ih_l1`, `weight_hh_l1`) відповідають за другий шар.### 🟡 Рівень 2 — ПлюсНавчи ту саму модель зі станом **32** і **256** замість 128, по одному зерну.Побудуй три точки «розмір стану → перплексія» і додай до них час навчання.**Зроблено, якщо:** є таблиця з трьох рядків і висновок словами про те, чи окупаєтьсязбільшення стану на цьому корпусі.### 🔴 Рівень 3 — ВикликЗаміряй, чи виживе перевага біграми на **довгих** реченнях. Відбери перевірніречення довжиною щонайменше 12 слів і порахуй перплексію RNN і найкращої біграми**тільки на них**.**Зроблено, якщо:** названо обидва числа з розкидом по трьох зернах і сказано, чирізниця більша за розкид. Підказка: розділ 11 уже показує, куди дивитись.